In [ ]:
import os
import json
import requests
import random
from datasets import load_dataset
from collections import defaultdict
from PIL import Image, UnidentifiedImageError
from io import BytesIO

random.seed(42)

# ─── CONFIG ───────────────────────────────────────────────────────────────────
TARGET_CATEGORIES = [
    "backpack", "bag", "calculator", "charger", "hairbrush",
    "headphones", "highlighter", "jar", "kettle", "keyboard",
    "laptop", "mouse", "notebook", "onion", "remote",
    "scissor", "stapler", "tape", "toothbrush", "umbrella",
    "wallet", "watch","apple", "book", "bottle", "bowl", "cup", "fork", "keys", "knife", "marker", "mug", "pen", "phone", "plate", "shoe", "spoon", "tomato",
    "tray", "basket", "pot", "pan","sock", "glove", "t-shirt","earbuds","egg","spatula"
]
MAX_PER_CATEGORY = 20
OUTPUT_DIR   = "/content/drive/MyDrive/eng521/Grasp Point Prediction/data/pixmo_subset_v3"
IMAGES_DIR   = os.path.join(OUTPUT_DIR, "images")
METADATA_PATH = os.path.join(OUTPUT_DIR, "metadata.json")
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(IMAGES_DIR, exist_ok=True)

# Load dataset in streaming mode
print("Loading PixMo-Points dataset (streaming)...")
dataset = load_dataset("allenai/pixmo-points", split="train", streaming=True)

# Track collected samples per category
collected = defaultdict(list)
done = set()  # categories that already hit 20

metadata = []
image_counters = defaultdict(int)

print("Streaming and filtering...")

for sample in dataset:
    label = sample.get("label", "").strip().lower()

    # skip if not a target category or already full
    if label not in TARGET_CATEGORIES:
        continue
    if label in done:
        continue

    # pointing method only
    if sample.get("collection_method") != "pointing":
        continue

    # must have exactly 1 point
    points = sample.get("points", [])
    if len(points) != 1:
        continue

    # download and validate image
    url = sample.get("image_url") or sample.get("url", "")
    if not url:
        continue

    try:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"  Skipping bad image for {label}: {e}")
        continue

    # save image
    image_counters[label] += 1
    image_name  = f"{label}_{image_counters[label]}.jpg"
    image_path  = os.path.join(IMAGES_DIR, image_name)
    img.save(image_path)

    # build metadata entry in same format as original
    entry = {
        "id": sample.get("id", sample.get("idx", len(metadata))),
        "label": label,
        "collection_method": "pointing",
        "count": None,
        "points": [{"x": p["x"], "y": p["y"]} for p in points],
        "image_path": image_path,
        "image_url": url
    }

    collected[label].append(entry)
    print(f"  [{label}] {len(collected[label])}/{MAX_PER_CATEGORY}")

    if len(collected[label]) >= MAX_PER_CATEGORY:
        done.add(label)
        print(f"  ✅ {label} complete")

    # stop early if all categories are done
    if len(done) == len(TARGET_CATEGORIES):
        print("All categories complete!")
        break

# Flatten and save
for label in TARGET_CATEGORIES:
    metadata.extend(collected[label])

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Done!")
print(f"Total samples: {len(metadata)}")
print("\nPer category:")
for label in TARGET_CATEGORIES:
    print(f"  {label}: {len(collected[label])}")
print(f"\nMetadata saved to: {METADATA_PATH}")
print(f"Images saved to:   {IMAGES_DIR}")

Loading PixMo-Points dataset (streaming)...


README.md: 0.00B [00:00, ?B/s]

Streaming and filtering...
  Skipping bad image for book: HTTPSConnectionPool(host='www.cbc.ca', port=443): Read timed out. (read timeout=10)
  [bowl] 1/20
  [headphones] 1/20
  [pot] 1/20
  [pot] 2/20
  [spoon] 1/20
  [book] 1/20
  Skipping bad image for glove: HTTPSConnectionPool(host='d1466nnw0ex81e.cloudfront.net', port=443): Max retries exceeded with url: /n_iv/600/4885117.jpg (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79a40c180350>: Failed to resolve 'd1466nnw0ex81e.cloudfront.net' ([Errno -5] No address associated with hostname)"))
  [bowl] 2/20
  [knife] 1/20
  [bottle] 1/20
  [pot] 3/20
  [spatula] 1/20
  [bowl] 3/20
  [pot] 4/20
  [pan] 1/20
  [book] 2/20
  [mug] 1/20
  [onion] 1/20
  [tomato] 1/20
  [bowl] 4/20
  [tomato] 2/20
  [pot] 5/20
  [watch] 1/20
  [knife] 2/20
  [watch] 2/20
  [cup] 1/20
  [phone] 1/20
  [jar] 1/20
  [pot] 6/20
  [plate] 1/20
  [tomato] 3/20
  [plate] 2/20
  Skipping bad image for laptop: HTTPSConnectionPool(host

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  [plate] 14/20
  Skipping bad image for bottle: cannot identify image file <_io.BytesIO object at 0x79a40c16c630>
  [phone] 5/20
  [cup] 8/20
  [cup] 9/20
  [keyboard] 6/20
  [phone] 6/20
  [tray] 1/20
  [plate] 15/20
  [basket] 6/20
  Skipping bad image for mug: broken data stream when reading image file
  Skipping bad image for watch: cannot identify image file <_io.BytesIO object at 0x79a53bd1fa60>
  [shoe] 5/20
  [sock] 2/20
  [sock] 3/20
  [bag] 2/20
  [spoon] 9/20
  [cup] 10/20
  [bag] 3/20
  Skipping bad image for plate: cannot identify image file <_io.BytesIO object at 0x79a40a0230b0>
  [knife] 14/20
  [plate] 16/20
  [basket] 7/20
  [plate] 17/20
  [pan] 6/20
  [umbrella] 2/20
  [fork] 8/20
  [book] 7/20
  [plate] 18/20
  [spoon] 10/20
  [bowl] 15/20
  [watch] 7/20
  [bowl] 16/20
  Skipping bad image for pen: HTTPSConnectionPool(host='static.cdn.asset.aparat.cloud', port=443): Read timed out. (read timeout=10)
  [shoe] 6/20
  [cup] 11/20
  Skipping bad image for cup: cannot i

In [ ]:
import os, json

# Count images on disk
images = os.listdir(IMAGES_DIR)
print(f"Total images on disk: {len(images)}")

# Count from metadata
with open(METADATA_PATH) as f:
    metadata = json.load(f)
print(f"Total metadata entries: {len(metadata)}")

# Per category breakdown
from collections import Counter
counts = Counter(item["label"] for item in metadata)
for label in TARGET_CATEGORIES:
    print(f"  {label}: {counts.get(label, 0)}/{MAX_PER_CATEGORY}")

Total images on disk: 914
Total metadata entries: 914
  backpack: 20/20
  bag: 20/20
  calculator: 20/20
  charger: 20/20
  hairbrush: 5/20
  headphones: 20/20
  highlighter: 4/20
  jar: 20/20
  kettle: 20/20
  keyboard: 20/20
  laptop: 20/20
  mouse: 20/20
  notebook: 20/20
  onion: 20/20
  remote: 20/20
  scissor: 18/20
  stapler: 15/20
  tape: 20/20
  toothbrush: 20/20
  umbrella: 20/20
  wallet: 20/20
  watch: 20/20
  apple: 20/20
  book: 20/20
  bottle: 20/20
  bowl: 20/20
  cup: 20/20
  fork: 20/20
  keys: 20/20
  knife: 20/20
  marker: 20/20
  mug: 20/20
  pen: 20/20
  phone: 20/20
  plate: 20/20
  shoe: 20/20
  spoon: 20/20
  tomato: 20/20
  tray: 20/20
  basket: 20/20
  pot: 20/20
  pan: 20/20
  sock: 20/20
  glove: 20/20
  t-shirt: 20/20
  earbuds: 12/20
  egg: 20/20
  spatula: 20/20


In [ ]:
import json
import random
import os
from collections import defaultdict, Counter

random.seed(42)

METADATA_PATH = "/content/drive/MyDrive/eng521/Grasp Point Prediction/data/pixmo_subset_v3/metadata.json"
OUTPUT_DIR    = "/content/drive/MyDrive/eng521/Grasp Point Prediction/data/pixmo_subset_v3"
SPLITS_DIR    = os.path.join(OUTPUT_DIR, "splits1")
os.makedirs(SPLITS_DIR, exist_ok=True)

with open(METADATA_PATH) as f:
    metadata = json.load(f)

by_category = defaultdict(list)
for item in metadata:
    by_category[item["label"]].append(item)

train, val, test = [], [], []

print(f"{'Category':<20} {'Total':>6} {'Train':>6} {'Val':>5} {'Test':>5}")
print("-" * 48)

for label, items in sorted(by_category.items()):
    random.shuffle(items)
    n = len(items)
    n_test  = max(1, round(n * 0.20))
    n_val   = max(1, round(n * 0.20))
    n_train = n - n_val - n_test

    train += items[:n_train]
    val   += items[n_train:n_train + n_val]
    test  += items[n_train + n_val:]

    print(f"{label:<20} {n:>6} {n_train:>6} {n_val:>5} {len(items[n_train+n_val:]):>5}")

print("-" * 48)
print(f"{'TOTAL':<20} {len(metadata):>6} {len(train):>6} {len(val):>5} {len(test):>5}")

# Save
for split_name, split_data in [("train", train), ("val", val), ("test", test)]:
    path = os.path.join(SPLITS_DIR, f"{split_name}.json")
    with open(path, "w") as f:
        json.dump(split_data, f, indent=2)
    print(f"\n✅ {split_name}.json — {len(split_data)} samples → {path}")

# ── Sanity checks ─────────────────────────────────────────────────────────────
print("\n── Sanity Checks ──")

# 1. No overlap between splits
train_urls = set(i["image_url"] for i in train)
val_urls   = set(i["image_url"] for i in val)
test_urls  = set(i["image_url"] for i in test)
assert not train_urls & val_urls,  "❌ Overlap between train and val!"
assert not train_urls & test_urls, "❌ Overlap between train and test!"
assert not val_urls   & test_urls, "❌ Overlap between val and test!"
print("✅ No overlap between splits")

# 2. All categories present in all splits
train_cats = set(i["label"] for i in train)
val_cats   = set(i["label"] for i in val)
test_cats  = set(i["label"] for i in test)
missing_val  = train_cats - val_cats
missing_test = train_cats - test_cats
print(f"✅ Categories missing from val:  {missing_val  or 'None'}")
print(f"✅ Categories missing from test: {missing_test or 'None'}")

# 3. Flag thin categories (< 4 test samples)
test_counts = Counter(i["label"] for i in test)
thin = {l: c for l, c in test_counts.items() if c < 4}
if thin:
    print(f"\n⚠️  Thin test categories (< 4 samples) — treat metrics with caution:")
    for label, count in sorted(thin.items()):
        print(f"   {label}: {count}")
else:
    print("✅ All categories have ≥ 4 test samples")

Category              Total  Train   Val  Test
------------------------------------------------
apple                    20     12     4     4
backpack                 20     12     4     4
bag                      20     12     4     4
basket                   20     12     4     4
book                     20     12     4     4
bottle                   20     12     4     4
bowl                     20     12     4     4
calculator               20     12     4     4
charger                  20     12     4     4
cup                      20     12     4     4
earbuds                  12      8     2     2
egg                      20     12     4     4
fork                     20     12     4     4
glove                    20     12     4     4
hairbrush                 5      3     1     1
headphones               20     12     4     4
highlighter               4      2     1     1
jar                      20     12     4     4
kettle                   20     12     4     4
keyboard   

AssertionError: ❌ Overlap between train and val!

In [ ]:
train_keys = set((i["image_path"], i["label"], str(i["points"])) for i in train)
val_keys   = set((i["image_path"], i["label"], str(i["points"])) for i in val)
test_keys  = set((i["image_path"], i["label"], str(i["points"])) for i in test)

assert not train_keys & val_keys,  "❌ Overlap between train and val!"
assert not train_keys & test_keys, "❌ Overlap between train and test!"
assert not val_keys   & test_keys, "❌ Overlap between val and test!"
print("✅ No overlap between splits")

✅ No overlap between splits


In [ ]:
import json

with open("/content/drive/MyDrive/eng521/Grasp Point Prediction/data/pixmo_subset_v3/metadata.json") as f:
    data = json.load(f)

# Fix IDs
for i, d in enumerate(data):
    d["id"] = i

with open("/content/drive/MyDrive/eng521/Grasp Point Prediction/data/pixmo_subset_v3/metadata.json", "w") as f:
    json.dump(data, f, indent=2)

print(f"Fixed IDs for {len(data)} samples")

Fixed IDs for 914 samples
